## Minesweeper: generate the numbers for safe squares in a Minesweeper grid

```text
Тебе даётся сетка (например, список списков) с минами (True/False или 1/0).
Ты должен:
    Пройти по каждой клетке.
    Если клетка содержит мину — оставить её как есть (обычно -1 или 9).
    Если клетка безопасная — посчитать, сколько мин находится в соседних 8 клетках (по горизонтали, вертикали и диагонали).
    Записать это число в эту клетку.
В итоге у тебя должен получиться DataFrame, где каждая ячейка — либо мина, либо число от 0 до 8.
```

In [251]:
import pandas as pd 
import numpy as np 

**51**. Let's suppose we're playing Minesweeper on a 5 by 4 grid, i.e.
```
X = 5
Y = 4
```
To begin, generate a DataFrame `df` with two columns, `'x'` and `'y'` containing every coordinate for this grid. That is, the DataFrame should start:
```
   x  y
0  0  0
1  0  1
2  0  2
...
```

In [252]:
X = 5
Y = 4

x = np.arange(X)
y = np.arange(Y)

idx = pd.MultiIndex.from_product([x, y])
df = idx.to_frame().reset_index(drop=True)
df.columns = ['x', 'y']

df

,x,y
0,0,0
1,0,1
2,0,2
3,0,3
4,1,0
5,1,1
6,1,2
7,1,3
8,2,0
9,2,1


**52**. For this DataFrame `df`, create a new column of zeros (safe) and ones (mine). The probability of a mine occuring at each location should be 0.4.

In [253]:
df['mine'] = np.random.choice([0, 1], size=df.shape[0], p=[.6, .4])
df 

,x,y,mine
0,0,0,1
1,0,1,1
2,0,2,0
3,0,3,1
4,1,0,1
5,1,1,0
6,1,2,0
7,1,3,0
8,2,0,1
9,2,1,0


**53**. Now create a new column for this DataFrame called `'adjacent'`. This column should contain the number of mines found on adjacent squares in the grid. 

(E.g. for the first row, which is the entry for the coordinate `(0, 0)`, count how many mines are found on the coordinates `(0, 1)`, `(1, 0)` and `(1, 1)`.)

In [254]:
pt = df.pivot_table(values='mine', columns='x', index='y', aggfunc='sum')

adj = (
    pt.shift().shift(axis=1) + pt.shift() +
    pt.shift().shift(-1, axis=1) + pt.shift(-1, axis=1) + 
    pt.shift(-1).shift(-1, axis=1) + pt.shift(-1) + 
    pt.shift(-1).shift(axis=1) + pt.shift(axis=1)
    ).fillna(0).astype(int)

adj = adj.stack()
df = df.merge(adj.reset_index(), on=['x', 'y'])

df = df.rename(columns={0: 'adjacent'})
df

,x,y,mine,adjacent
0,0,0,1,0
1,0,1,1,0
2,0,2,0,0
3,0,3,1,0
4,1,0,1,0
5,1,1,0,5
6,1,2,0,4
7,1,3,0,0
8,2,0,1,0
9,2,1,0,5


**54**. For rows of the DataFrame that contain a mine, set the value in the `'adjacent'` column to NaN.

In [255]:
df.loc[df.mine == 1, 'adjacent'] = np.nan 
df 

,x,y,mine,adjacent
0,0,0,1,NaN
1,0,1,1,NaN
2,0,2,0,0.0
3,0,3,1,NaN
4,1,0,1,NaN
5,1,1,0,5.0
6,1,2,0,4.0
7,1,3,0,0.0
8,2,0,1,NaN
9,2,1,0,5.0


**55**. Finally, convert the DataFrame to grid of the adjacent mine counts: columns are the `x` coordinate, rows are the `y` coordinate.

In [256]:
df.pivot_table(index='y', columns='x', values='adjacent')

x,0,1,2,3,4
y,,,,,
0,NaN,NaN,NaN,0.0,NaN
1,NaN,5.0,5.0,NaN,0.0
2,0.0,4.0,NaN,NaN,0.0
3,NaN,0.0,NaN,NaN,NaN
